# DO.md #1 — Sanity Check

**목적:** `val/` 폴더(밝은 이미지)로 학습 → test 각도에서 밝은 이미지가 잘 나오는지 확인

**확인 항목:** 3DGS 엔진 자체가 정상 동작하는지 검증 (조명 문제와 무관하게)

## Cell 1. Google Drive 마운트

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Cell 2. 경로 설정

> **주의:** Drive에서 3DRR 폴더 위치가 다르면 `DRIVE_ROOT` 를 수정하세요.

In [3]:
import os

# ===== 경로 설정 (필요시 수정) =====
DRIVE_ROOT = '/content/drive/MyDrive/3DRR'
# ==================================

CODEBASE_DIR = os.path.join(DRIVE_ROOT, '3DRR_codebase')
DATA_DIR     = os.path.join(DRIVE_ROOT, 'validation', 'BlueHawaii')
CONFIG_PATH  = os.path.join(CODEBASE_DIR, 'config', 'sanity_check.yaml')

print('CODEBASE_DIR :', CODEBASE_DIR)
print('DATA_DIR     :', DATA_DIR)
print('CONFIG_PATH  :', CONFIG_PATH)

# 경로 존재 확인
for path in [CODEBASE_DIR, DATA_DIR]:
    status = '✓' if os.path.exists(path) else '✗ NOT FOUND'
    print(f'  {status}  {path}')

CODEBASE_DIR : /content/drive/MyDrive/3DRR/3DRR_codebase
DATA_DIR     : /content/drive/MyDrive/3DRR/validation/BlueHawaii
CONFIG_PATH  : /content/drive/MyDrive/3DRR/3DRR_codebase/config/sanity_check.yaml
  ✓  /content/drive/MyDrive/3DRR/3DRR_codebase
  ✓  /content/drive/MyDrive/3DRR/validation/BlueHawaii


## Cell 3. 패키지 설치

In [4]:
%pip install -q gsplat torchvision omegaconf tqdm PyYAML

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 9.1 MB/s eta 0:00:00


## Cell 4. Sanity Check용 Config 생성

- `DATA_PATH`: BlueHawaii (Drive 경로)
- `split=val` → 밝은 이미지로 학습
- 빠른 검증을 위해 `TRAIN_TOTAL_STEP: 5000` 으로 축소

In [5]:
import yaml

config = {
    'DATASET': {
        'NAME': 'BlueHawaii',
        'DATA_PATH': DATA_DIR,
        'BACKGROUND_COLOR': 255.0,
    },
    'MODEL': {
        'NAME': 'Simple3DGS',
        # Gaussian initialization
        'NUM_INIT_POINTS': 100000,
        'SH_DEGREE': 3,
        'SCENE_SCALE': 2.0,
        # Training (빠른 sanity check용: 5000 steps)
        'TRAIN_TOTAL_STEP': 5000,
        'VAL_INTERVAL_STEP': 1000,
        'LOG_INTERVAL_STEP': 100,
        'SH_UPGRADE_INTERVAL': 1000,
        # Learning rates
        'LR_MEANS': 1.6e-4,
        'LR_MEANS_FINAL': 1.6e-6,
        'LR_QUATS': 1.0e-3,
        'LR_SCALES': 5.0e-3,
        'LR_OPACITIES': 5.0e-2,
        'LR_SH0': 2.5e-3,
        'LR_SHN': 1.25e-4,
        # Densification
        'DENSIFY_START_STEP': 500,
        'DENSIFY_STOP_STEP': 3000,
        'DENSIFY_INTERVAL': 100,
        'DENSIFY_GRAD_THRESH': 0.0002,
        'OPACITY_RESET_INTERVAL': 3000,
        # Loss
        'LAMBDA_SSIM': 0.2,
    }
}

os.makedirs(os.path.dirname(CONFIG_PATH), exist_ok=True)
with open(CONFIG_PATH, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f'Config saved to: {CONFIG_PATH}')

Config saved to: /content/drive/MyDrive/3DRR/3DRR_codebase/config/sanity_check.yaml


## Cell 5. 학습 실행

> `train.py`는 `3DRR_codebase/` 를 working directory로 가정하므로 `os.chdir` 로 이동합니다.

In [ ]:
import sys
import os

# Check if CODEBASE_DIR exists before trying to change to it
if not os.path.exists(CODEBASE_DIR):
    print(f"Error: CODEBASE_DIR '{CODEBASE_DIR}' does not exist.")
    print("Please ensure the '3DRR_codebase' folder is uploaded to your Google Drive")
    print("at the path specified by DRIVE_ROOT in Cell 2, or adjust DRIVE_ROOT if needed.")
    # Raise an error to prevent further execution that will fail
    raise FileNotFoundError(f"CODEBASE_DIR '{CODEBASE_DIR}' not found.")

# working directory를 codebase로 변경 (core/ 모듈 import를 위해 필수)
os.chdir(CODEBASE_DIR)
if CODEBASE_DIR not in sys.path:
    sys.path.insert(0, CODEBASE_DIR)

print(f'Working directory: {os.getcwd()}')

# train 실행
from train import train
train(CONFIG_PATH)

Working directory: /content/drive/MyDrive/3DRR/3DRR_codebase
DATASET:
    BACKGROUND_COLOR:          255.0
    DATA_PATH:                 /content/drive/MyDrive/3DRR/validation/BlueHawaii
    NAME:                      BlueHawaii
MODEL:
    DENSIFY_GRAD_THRESH:       0.0002
    DENSIFY_INTERVAL:          100
    DENSIFY_START_STEP:        500
    DENSIFY_STOP_STEP:         3000
    LAMBDA_SSIM:               0.2
    LOG_INTERVAL_STEP:         100
    LR_MEANS:                  0.00016
    LR_MEANS_FINAL:            1.6e-06
    LR_OPACITIES:              0.05
    LR_QUATS:                  0.001
    LR_SCALES:                 0.005
    LR_SH0:                    0.0025
    LR_SHN:                    0.000125
    NAME:                      Simple3DGS
    NUM_INIT_POINTS:           100000
    OPACITY_RESET_INTERVAL:    3000
    SCENE_SCALE:               2.0
    SH_DEGREE:                 3
    SH_UPGRADE_INTERVAL:       1000
    TRAIN_TOTAL_STEP:          5000
    VAL_INTERVAL_STEP:     

  0%|          | 0/5000 [00:00<?, ?it/s]

Output()

gsplat: CUDA extension has been set up successfully in 781.18 seconds.

  1%|          | 49/5000 [13:07<06:33, 12.58it/s, loss=0.8767, n_gs=1e+5, psnr=1.02]

## Cell 6. 결과 확인

학습 중 저장된 validation 렌더링 이미지를 인라인으로 표시합니다.

In [ ]:
import glob
from IPython.display import display, Image as IPImage

outputs_root = os.path.join(CODEBASE_DIR, 'outputs')

# 가장 최근 실험 폴더 탐색
exp_dirs = sorted(glob.glob(os.path.join(outputs_root, '**', 'examples'), recursive=True))
if not exp_dirs:
    print('outputs/ 폴더를 찾을 수 없습니다. 학습이 완료됐는지 확인하세요.')
else:
    latest_examples = exp_dirs[-1]
    print(f'결과 폴더: {latest_examples}')

    # val 렌더링 이미지 표시
    val_imgs = sorted(glob.glob(os.path.join(latest_examples, 'val_step*.jpg')))
    if val_imgs:
        print(f'\nValidation 렌더링 ({len(val_imgs)}개):')
        for img_path in val_imgs:
            print(f'  {os.path.basename(img_path)}')
            display(IPImage(img_path, width=600))
    else:
        print('val 렌더링 이미지가 없습니다.')

    # train augment 확인
    train_aug = os.path.join(latest_examples, 'train_aug.jpg')
    if os.path.exists(train_aug):
        print('\nTrain 입력 이미지 샘플 (val/ 폴더 = 밝은 이미지여야 함):')
        display(IPImage(train_aug, width=600))

## Cell 7. 수치 평가 (PSNR / SSIM / LPIPS)

`core/evaluate.py` 를 사용해 렌더링 결과와 GT를 비교합니다.
결과는 `outputs/.../metrics.json` 에 저장됩니다.

In [ ]:
%pip install -q lpips torchmetrics

import glob, os, sys

# CODEBASE_DIR이 path에 있는지 확인
if CODEBASE_DIR not in sys.path:
    sys.path.insert(0, CODEBASE_DIR)

from core.evaluate import compute_metrics, save_metrics, print_metrics

# 가장 최근 실험 output 폴더 자동 탐색
outputs_root = os.path.join(CODEBASE_DIR, 'outputs')
test_dirs = sorted(glob.glob(os.path.join(outputs_root, '**', 'test'), recursive=True))
if not test_dirs:
    print('test 결과 폴더를 찾을 수 없습니다. 학습이 완료됐는지 확인하세요.')
else:
    pred_dir   = test_dirs[-1]                   # 최신 실험
    output_dir = os.path.dirname(pred_dir)       # outputs/<exp>/<time>/
    gt_dir     = os.path.join(DATA_DIR, 'test')  # GT 이미지

    print(f'pred_dir : {pred_dir}')
    print(f'gt_dir   : {gt_dir}')

    metrics = compute_metrics(pred_dir, gt_dir)
    save_metrics(metrics, output_dir)
    print_metrics(metrics)